# Model training

Since our data is ready to use, we can train machine learning models to detect heart attack risk.

Here, we will train three different models from Scikit-learn:
* Logistic Regression
* Random Forest Classifier
* K-Nearest Neighbors

Our goal is to find the model which delivers the best result to solve this problem.

## Load data and split into training and testing

First, we load our processed data and split it.

In [62]:
import sys
sys.path.append("../src")

import data.make_dataset as data
import features.build_features as features
import models.train as train

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

import importlib
importlib.reload(data)
importlib.reload(features)
importlib.reload(train)

<module 'models.train' from 'c:\\Users\\isaac\\Documents\\GitHub\\heart-attack-risk-detection\\notebooks\\../src\\models\\train.py'>

In [45]:
# Load dataset
df = data.load_dataset("../data/processed/dataset-processed.csv")
df.head()

,age,gender,chestpain,restingBP,serumcholestrol,fastingbloodsugar,restingrelectro,maxheartrate,exerciseangia,oldpeak,slope,noofmajorvessels,target
0,53,1,2,171,0,0,1,147,0,5.3,3,3,1
1,40,1,0,94,229,0,1,115,0,3.7,1,1,0
2,49,1,2,133,142,0,0,202,1,5.0,1,0,0
3,43,1,0,138,295,1,1,153,0,3.2,2,2,1
4,31,1,1,199,0,0,2,136,0,5.3,3,2,1


In [48]:
# Split into X & y
X, y = features.split_features(df)

In [49]:
X

,age,gender,chestpain,restingBP,serumcholestrol,fastingbloodsugar,restingrelectro,maxheartrate,exerciseangia,oldpeak,slope,noofmajorvessels
0,53,1,2,171,0,0,1,147,0,5.3,3,3
1,40,1,0,94,229,0,1,115,0,3.7,1,1
2,49,1,2,133,142,0,0,202,1,5.0,1,0
3,43,1,0,138,295,1,1,153,0,3.2,2,2
4,31,1,1,199,0,0,2,136,0,5.3,3,2
...,...,...,...,...,...,...,...,...,...,...,...,...
995,48,1,2,139,349,0,2,183,1,5.6,2,2
996,47,1,3,143,258,1,1,98,1,5.7,1,0
997,69,1,0,156,434,1,0,196,0,1.4,3,1
998,45,1,1,186,417,0,1,117,1,5.9,3,2


In [50]:
y

0      1
1      0
2      0
3      1
4      1
      ..
995    1
996    0
997    1
998    1
999    0
Name: target, Length: 1000, dtype: int64

## Split into training and testing

In [51]:
X_train, X_test, y_train, y_test = train.split_train_test(X, y)

In [52]:
X_train.shape, y_train.shape

((800, 12), (800,))

In [53]:
X_test.shape, y_test.shape

((200, 12), (200,))

## Model creation

Now, we instantiate the models to use.

In [64]:
models = {
    "Random Forest": RandomForestClassifier(),
    "Logistic Regression": LogisticRegression(max_iter=100),
    "KNN": KNeighborsClassifier()
}

## Cross-validation

Now, we take our models and perform cross-validation. This, to have an idea of which model performs better. Also, we weant to identify whether a model needs some tuning process.

In [58]:
cv_scorings = train.perform_cross_validation(models, X_train, y_train)
cv_scorings

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    5.9s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    3.7s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    3.8s finished


{'Random Forest': array([0.9625 , 0.99375, 0.95625, 0.95625, 0.9875 ]),
 'Logistic Regression': array([0.93125, 0.95   , 0.91875, 0.91875, 0.94375]),
 'KNN': array([0.76875, 0.825  , 0.83125, 0.7625 , 0.8125 ])}

## Fine tuning

After the cross-validation, we can see both Random Forest and Logistic Regression present a good performance. On the other hand, KNN has low accuracies, compared to the two previous models.

Let's try to improve KNN through some tuning using grid search.

In [57]:
new_knn = KNeighborsClassifier()

knn_grid = {
    "n_neighbors": [3, 5, 7, 9, 11, 15, 21],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan", "minkowski"],
    "p": [1, 2]
}

knn_enhanced = train.apply_grid_search(new_knn, X_train, y_train, knn_grid)

Best parameters: {'metric': 'manhattan', 'n_neighbors': 21, 'p': 1, 'weights': 'distance'}
Best accuracy: 0.8487500000000001


After performing some grid search, we could barely improve the best accuracy obtained during the cross-validation. Let's continue with this model

In [65]:
models["KNN"] = knn_enhanced

## Train models

Now, let's train each model.

In [66]:
train.train_models(models, X_train, y_train)

c:\Users\isaac\Documents\curso_ml\project_heart_disease\env\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
